In [1]:
# ==========================================================
# Project Setup
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np

# This notebook must remain inside the /notebooks folder
PROJECT_ROOT = Path.cwd().parent

# Data folders
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

# Results folders for data preparation
RESULTS_DIR = PROJECT_ROOT / "results"
PREPARATION_RESULTS_DIR = RESULTS_DIR / "data_preparation"
TABLES_DIR = PREPARATION_RESULTS_DIR / "tables"
FIGURES_DIR = PREPARATION_RESULTS_DIR / "figures"

# Create output folders automatically
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Results:", PREPARATION_RESULTS_DIR)

Project root: /Users/mas364/Desktop/AI_Crypto_Forecasting
Raw data: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/raw
Processed data: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed
Results: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/data_preparation


## Import libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

## Setting file paths

In [3]:
# ==========================================================
# Input File Paths
# ==========================================================

btc_path = RAW_DATA_DIR / "btc-usd-max.csv"
eth_path = RAW_DATA_DIR / "eth-usd-max.csv"
trends_path = RAW_DATA_DIR / "google_trends.csv"
fear_greed_path = RAW_DATA_DIR / "fear_greed.csv"

# Confirm that all required files exist
required_files = {
    "Bitcoin": btc_path,
    "Ethereum": eth_path,
    "Google Trends": trends_path,
    "Fear & Greed": fear_greed_path
}

missing_files = [
    str(path)
    for path in required_files.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
    )

print("All required raw data files were found.")

All required raw data files were found.


## Loading datasets

In [4]:
btc = pd.read_csv(btc_path)
eth = pd.read_csv(eth_path)
google_trends = pd.read_csv(trends_path)
fear_greed = pd.read_csv(fear_greed_path)

datasets = {
    "Bitcoin": btc,
    "Ethereum": eth,
    "Google Trends": google_trends,
    "Fear & Greed": fear_greed
}

for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    display(df.head())

Bitcoin


,event_date,close_price_usd,market_cap_usd,volume_usd
0,2013-04-28 00:00:00 UTC,135.30,1.500518e+09,0.0
1,2013-04-29 00:00:00 UTC,141.96,1.575032e+09,0.0
2,2013-04-30 00:00:00 UTC,135.30,1.501657e+09,0.0
3,2013-05-01 00:00:00 UTC,117.00,1.298952e+09,0.0
4,2013-05-02 00:00:00 UTC,103.43,1.148668e+09,0.0


Ethereum


,event_date,close_price_usd,market_cap_usd,volume_usd
0,2015-08-07 00:00:00 UTC,2.831620,0.000000e+00,9.062200e+04
1,2015-08-08 00:00:00 UTC,1.330750,8.033948e+07,3.680700e+05
2,2015-08-10 00:00:00 UTC,0.687586,4.155631e+07,4.004641e+05
3,2015-08-11 00:00:00 UTC,1.067379,6.453901e+07,1.518998e+06
4,2015-08-12 00:00:00 UTC,1.256613,7.601326e+07,2.073893e+06


Google Trends


,Date,Bitcoin,Ethereum
0,2021-06-01,32,34
1,2021-06-02,31,33
2,2021-06-03,31,33
3,2021-06-04,35,31
4,2021-06-05,31,28


Fear & Greed


,Date,Fear_Greed_Index,Classification
0,2021-06-01,20,Extreme Fear
1,2021-06-02,23,Extreme Fear
2,2021-06-03,24,Extreme Fear
3,2021-06-04,27,Fear
4,2021-06-05,24,Extreme Fear


# Data Cleaning
* Convert all date columns to datetime.
* Remove timezone information.
* Standardise date column names.
* Filter datasets to the common study period (1 June 2021 – 6 May 2026).

### Convert Date Columns to Datetime

The date columns were converted to the `datetime` data type to ensure a consistent format for filtering, merging, and time-series analysis.

In [5]:
# Convert date columns to datetime

btc["event_date"] = pd.to_datetime(btc["event_date"], utc=True)
eth["event_date"] = pd.to_datetime(eth["event_date"], utc=True)

google_trends["Date"] = pd.to_datetime(google_trends["Date"])
fear_greed["Date"] = pd.to_datetime(fear_greed["Date"])

# Display data types
print("Bitcoin:", btc["event_date"].dtype)
print("Ethereum:", eth["event_date"].dtype)
print("Google Trends:", google_trends["Date"].dtype)
print("Fear & Greed:", fear_greed["Date"].dtype)





Bitcoin: datetime64[us, UTC]
Ethereum: datetime64[us, UTC]
Google Trends: datetime64[us]
Fear & Greed: datetime64[us]


### Interpretation
The output confirms that all date columns have been successfully converted to the `datetime` format. The Bitcoin and Ethereum datasets initially retain UTC timezone information, while the Google Trends and Fear & Greed datasets are stored as timezone-independent datetime values.

### Remove timezone information from cryptocurrency datasets

In [6]:

btc["event_date"] = btc["event_date"].dt.tz_localize(None)
eth["event_date"] = eth["event_date"].dt.tz_localize(None)

# Display the updated data types

print("=" * 50)
print("Bitcoin Dataset")
print("=" * 50)
print("Date Column Data Type:", btc["event_date"].dtype)

print("\n" + "=" * 50)
print("Ethereum Dataset")
print("=" * 50)
print("Date Column Data Type:", eth["event_date"].dtype)

Bitcoin Dataset
Date Column Data Type: datetime64[us]

Ethereum Dataset
Date Column Data Type: datetime64[us]


### Interpretation
The timezone information has been successfully removed from the Bitcoin and Ethereum datasets. Both date columns are now stored as timezone-independent datetime values, ensuring consistency with the remaining datasets before merging.

### Standardise Date Column Names

The date column names are standardised to ensure consistency across all datasets before merging.

In [7]:
# Rename date columns

btc = btc.rename(columns={"event_date": "Date"})
eth = eth.rename(columns={"event_date": "Date"})
fear_greed = fear_greed.rename(columns={"Date": "Date"})

# Display updated column names

print("=" * 50)
print("Bitcoin Dataset")
print("=" * 50)
print(btc.columns.tolist())

print("\n" + "=" * 50)
print("Ethereum Dataset")
print("=" * 50)
print(eth.columns.tolist())

print("\n" + "=" * 50)
print("Google Trends Dataset")
print("=" * 50)
print(google_trends.columns.tolist())

print("\n" + "=" * 50)
print("Fear & Greed Dataset")
print("=" * 50)
print(fear_greed.columns.tolist())

Bitcoin Dataset
['Date', 'close_price_usd', 'market_cap_usd', 'volume_usd']

Ethereum Dataset
['Date', 'close_price_usd', 'market_cap_usd', 'volume_usd']

Google Trends Dataset
['Date', 'Bitcoin', 'Ethereum']

Fear & Greed Dataset
['Date', 'Fear_Greed_Index', 'Classification']


### Interpretation

The date column names have been standardised across all datasets by renaming them to **Date**. Using a consistent column name simplifies the merging process and ensures that all datasets can be joined using a common key.

In [8]:
# Define the common study period

start_date = "2021-06-01"
end_date = "2026-05-06"

# Filter each dataset

btc = btc[(btc["Date"] >= start_date) & (btc["Date"] <= end_date)]
eth = eth[(eth["Date"] >= start_date) & (eth["Date"] <= end_date)]
google_trends = google_trends[(google_trends["Date"] >= start_date) & (google_trends["Date"] <= end_date)]
fear_greed = fear_greed[(fear_greed["Date"] >= start_date) & (fear_greed["Date"] <= end_date)]

# Display the filtered date ranges

print("=" * 50)
print("Bitcoin Dataset")
print("=" * 50)
print("Date Range:", btc["Date"].min().date(), "to", btc["Date"].max().date())

print("\n" + "=" * 50)
print("Ethereum Dataset")
print("=" * 50)
print("Date Range:", eth["Date"].min().date(), "to", eth["Date"].max().date())

print("\n" + "=" * 50)
print("Google Trends Dataset")
print("=" * 50)
print("Date Range:", google_trends["Date"].min().date(), "to", google_trends["Date"].max().date())

print("\n" + "=" * 50)
print("Fear & Greed Dataset")
print("=" * 50)
print("Date Range:", fear_greed["Date"].min().date(), "to", fear_greed["Date"].max().date())

Bitcoin Dataset
Date Range: 2021-06-01 to 2026-05-06

Ethereum Dataset
Date Range: 2021-06-01 to 2026-05-06

Google Trends Dataset
Date Range: 2021-06-01 to 2026-05-06

Fear & Greed Dataset
Date Range: 2021-06-01 to 2026-05-06


### Interpretation

The filtering process successfully aligned all datasets to the selected study period beginning on 1 June 2021. All four datasets were filtered to end on 6 May 2026, matching the shorter coverage of the Google Trends dataset identified during data understanding. Consequently, the final merged dataset spans the full common period shared by all four datasets, ensuring that every observation contains complete information.

### Sort the Datasets by Date

The datasets are sorted in chronological order to ensure that observations are correctly ordered before they are merged. This is an essential requirement for time-series analysis.

In [9]:
# Sort each dataset by date

btc = btc.sort_values("Date").reset_index(drop=True)
eth = eth.sort_values("Date").reset_index(drop=True)
google_trends = google_trends.sort_values("Date").reset_index(drop=True)
fear_greed = fear_greed.sort_values("Date").reset_index(drop=True)

# Display the first five dates

print("=" * 50)
print("Bitcoin Dataset")
print("=" * 50)
display(btc[["Date"]].head())

print("=" * 50)
print("Ethereum Dataset")
print("=" * 50)
display(eth[["Date"]].head())

print("=" * 50)
print("Google Trends Dataset")
print("=" * 50)
display(google_trends[["Date"]].head())

print("=" * 50)
print("Fear & Greed Dataset")
print("=" * 50)
display(fear_greed[["Date"]].head())

Bitcoin Dataset


,Date
0,2021-06-01
1,2021-06-02
2,2021-06-03
3,2021-06-04
4,2021-06-05


Ethereum Dataset


,Date
0,2021-06-01
1,2021-06-02
2,2021-06-03
3,2021-06-04
4,2021-06-05


Google Trends Dataset


,Date
0,2021-06-01
1,2021-06-02
2,2021-06-03
3,2021-06-04
4,2021-06-05


Fear & Greed Dataset


,Date
0,2021-06-01
1,2021-06-02
2,2021-06-03
3,2021-06-04
4,2021-06-05


### Interpretation

The datasets have been successfully sorted in chronological order. This ensures that the observations are correctly arranged for dataset integration and subsequent time-series analysis.

### Save the Cleaned Datasets

The cleaned datasets are saved after completing the data cleaning stage. This improves reproducibility and provides clean datasets for the data integration stage.

In [10]:
# ==========================================================
# Save Cleaned Datasets
# ==========================================================

cleaned_files = {
    "Bitcoin": (
        btc,
        PROCESSED_DATA_DIR / "bitcoin_clean.csv"
    ),
    "Ethereum": (
        eth,
        PROCESSED_DATA_DIR / "ethereum_clean.csv"
    ),
    "Google Trends": (
        google_trends,
        PROCESSED_DATA_DIR / "google_trends_clean.csv"
    ),
    "Fear & Greed": (
        fear_greed,
        PROCESSED_DATA_DIR / "fear_greed_clean.csv"
    )
}

for name, (dataset, output_path) in cleaned_files.items():
    dataset.to_csv(output_path, index=False)
    print(f"Saved {name}: {output_path}")

Saved Bitcoin: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/bitcoin_clean.csv
Saved Ethereum: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/ethereum_clean.csv
Saved Google Trends: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/google_trends_clean.csv
Saved Fear & Greed: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/fear_greed_clean.csv


## Data Integration
After completing the data cleaning stage, the next step is to integrate the four datasets into a single analytical dataset. Data integration combines the cryptocurrency market data, Google Trends search interest, and the Crypto Fear & Greed Index using the common **Date** column. This creates a unified dataset that will be used for exploratory data analysis, feature engineering, and forecasting model development.

In [11]:
# Rename Bitcoin columns

btc = btc.rename(columns={
    "close_price_usd": "Bitcoin_Price",
    "market_cap_usd": "Bitcoin_Market_Cap",
    "volume_usd": "Bitcoin_Volume"
})

# Rename Ethereum columns

eth = eth.rename(columns={
    "close_price_usd": "Ethereum_Price",
    "market_cap_usd": "Ethereum_Market_Cap",
    "volume_usd": "Ethereum_Volume"
})

# Rename Google Trends columns

google_trends = google_trends.rename(columns={
    "Bitcoin": "Bitcoin_Search_Interest",
    "Ethereum": "Ethereum_Search_Interest"
})

# Rename Fear & Greed columns

fear_greed = fear_greed.rename(columns={
    "Fear_Greed_Index": "Fear_Greed_Index",
    "Classification": "Market_Sentiment"
})

# Display updated column names

print("Bitcoin Columns")
print(btc.columns.tolist())

print("\nEthereum Columns")
print(eth.columns.tolist())

print("\nGoogle Trends Columns")
print(google_trends.columns.tolist())

print("\nFear & Greed Columns")
print(fear_greed.columns.tolist())

Bitcoin Columns
['Date', 'Bitcoin_Price', 'Bitcoin_Market_Cap', 'Bitcoin_Volume']

Ethereum Columns
['Date', 'Ethereum_Price', 'Ethereum_Market_Cap', 'Ethereum_Volume']

Google Trends Columns
['Date', 'Bitcoin_Search_Interest', 'Ethereum_Search_Interest']

Fear & Greed Columns
['Date', 'Fear_Greed_Index', 'Market_Sentiment']


### Merge the Datasets
The cleaned datasets are merged using an inner join on the **Date** column. An inner join ensures that only dates available across all datasets are retained, resulting in a consistent and complete dataset for subsequent analysis.

In [12]:
# Merge Bitcoin and Ethereum datasets

merged_data = btc.merge(
    eth,
    on="Date",
    how="inner",
    suffixes=("_btc", "_eth")
)

# Merge Google Trends data

merged_data = merged_data.merge(
    google_trends,
    on="Date",
    how="inner"
)

# Merge Fear & Greed data

merged_data = merged_data.merge(
    fear_greed,
    on="Date",
    how="inner"
)

# Display merged dataset information

print("=" * 50)
print("Merged Dataset")
print("=" * 50)

print("Rows:", merged_data.shape[0])
print("Columns:", merged_data.shape[1])
print("Date Range:", merged_data["Date"].min().date(), "to", merged_data["Date"].max().date())

display(merged_data.head())

Merged Dataset
Rows: 1799
Columns: 11
Date Range: 2021-06-01 to 2026-05-06


,Date,Bitcoin_Price,Bitcoin_Market_Cap,Bitcoin_Volume,Ethereum_Price,Ethereum_Market_Cap,Ethereum_Volume,Bitcoin_Search_Interest,Ethereum_Search_Interest,Fear_Greed_Index,Market_Sentiment
0,2021-06-01,36680.068202,6.887264e+11,3.353691e+10,2632.656600,3.064888e+11,3.989865e+10,32,34,20,Extreme Fear
1,2021-06-02,37685.717982,7.045313e+11,3.124069e+10,2717.154037,3.153133e+11,3.914886e+10,31,33,23,Extreme Fear
2,2021-06-03,39151.316184,7.331507e+11,3.339672e+10,2858.276702,3.339646e+11,3.810466e+10,31,33,24,Extreme Fear
3,2021-06-04,36938.720311,6.917475e+11,3.934549e+10,2694.497667,3.129911e+11,3.999658e+10,35,31,27,Fear
4,2021-06-05,35546.985965,6.629135e+11,3.406817e+10,2624.768915,3.027434e+11,3.473198e+10,31,28,24,Extreme Fear


In [13]:
# ==========================================================
# Validate and Repair Implausible Trading Volume Values
# ==========================================================
# During data quality assessment, a small number of trading
# volume observations were found to be substantially larger
# than the corresponding market capitalisation. Since daily
# trading volume should not greatly exceed the total market
# value of the asset, these observations were treated as
# source-data anomalies rather than genuine market activity.
#
# Implausible values are identified using the ratio:
#
#     Trading Volume / Market Capitalisation > 10
# Only the affected volume values are replaced with missing
# values. All other variables remain unchanged. These missing
# values are left as-is here and handled explicitly at the
# feature-engineering stage, where the appropriate treatment
# can be chosen per model (e.g. forward-fill only, to avoid
# introducing future information into the test period).

VOLUME_RATIO_LIMIT = 10

volume_columns = [
    ("Bitcoin_Volume", "Bitcoin_Market_Cap"),
    ("Ethereum_Volume", "Ethereum_Market_Cap"),
]

# Ensure observations are ordered chronologically
merged_data = merged_data.sort_values("Date").copy()

for volume_column, market_cap_column in volume_columns:

    volume_ratio = (
        merged_data[volume_column]
        / merged_data[market_cap_column]
    )

    invalid_mask = volume_ratio > VOLUME_RATIO_LIMIT

    print(f"\n{volume_column}")
    print(f"Implausible observations detected: {invalid_mask.sum()}")

    if invalid_mask.any():
        print("Affected dates:")
        print(
            merged_data.loc[
                invalid_mask,
                "Date"
            ].dt.date.tolist()
        )

    # Replace implausible values with missing values
    merged_data.loc[
        invalid_mask,
        volume_column
    ] = np.nan


# Final validation
print("\nRemaining missing values after volume repair:")

print(
    merged_data[
        ["Bitcoin_Volume", "Ethereum_Volume"]
    ].isnull().sum()
)


Bitcoin_Volume
Implausible observations detected: 9
Affected dates:
[datetime.date(2026, 4, 1), datetime.date(2026, 4, 2), datetime.date(2026, 4, 3), datetime.date(2026, 4, 4), datetime.date(2026, 4, 5), datetime.date(2026, 4, 6), datetime.date(2026, 4, 7), datetime.date(2026, 4, 8), datetime.date(2026, 4, 9)]

Ethereum_Volume
Implausible observations detected: 9
Affected dates:
[datetime.date(2026, 4, 1), datetime.date(2026, 4, 2), datetime.date(2026, 4, 3), datetime.date(2026, 4, 4), datetime.date(2026, 4, 5), datetime.date(2026, 4, 6), datetime.date(2026, 4, 7), datetime.date(2026, 4, 8), datetime.date(2026, 4, 9)]

Remaining missing values after volume repair:
Bitcoin_Volume     9
Ethereum_Volume    9
dtype: int64


## Data Quality Checks After Merge

Following dataset integration, the merged dataset is assessed to identify any data quality issues introduced during the merging process. This includes checking for missing values, missing dates, duplicate dates, and handling any missing values to ensure that the final analytical dataset is complete and suitable for exploratory data analysis and forecasting.

### Checking Missing Values

The merged dataset is examined for missing values to identify whether the integration process has introduced any incomplete observations.

In [14]:
missing_values = pd.DataFrame({
    "Missing Values": merged_data.isnull().sum(),
    "Percentage (%)": (merged_data.isnull().mean() * 100).round(2)
})

display(missing_values)

,Missing Values,Percentage (%)
Date,0,0.0
Bitcoin_Price,0,0.0
Bitcoin_Market_Cap,0,0.0
Bitcoin_Volume,9,0.5
Ethereum_Price,0,0.0
Ethereum_Market_Cap,0,0.0
Ethereum_Volume,9,0.5
Bitcoin_Search_Interest,0,0.0
Ethereum_Search_Interest,0,0.0
Fear_Greed_Index,0,0.0


###  Interpretation

The merged dataset contains 18 missing values, confined to Bitcoin_Volume and Ethereum_Volume (9 each, 0.5%). These are not gaps introduced by the merge — they are the result of the volume repair applied immediately after merging, which removed nine days (1–9 April 2026) of implausible trading-volume figures traced to a CoinGecko export error. No other variable contains missing values.

### Check for Missing Dates

The merged dataset is checked for missing calendar dates to ensure that the time series remains continuous throughout the study period.

In [15]:
# Create complete daily date range

date_range = pd.date_range(
    start=merged_data["Date"].min(),
    end=merged_data["Date"].max(),
    freq="D"
)

# Identify missing dates

missing_dates = date_range.difference(merged_data["Date"])

print("Number of Missing Dates:", len(missing_dates))

display(pd.DataFrame({"Missing Dates": missing_dates}))

Number of Missing Dates: 2


,Missing Dates
0,2022-05-23
1,2024-10-26


### Interpretation

The merged dataset contains **two missing calendar dates**, indicating that the time series is almost complete over the study period. These missing dates will be further investigated to determine whether they originate from the original source datasets or were introduced during the data integration process.

### Check which datasets contain the missing dates

In [16]:
# Check which datasets contain the missing dates

missing_dates_list = ["2022-05-23", "2024-10-26"]
missing_dates_list = pd.to_datetime(missing_dates_list)

for date in missing_dates_list:
    print("=" * 50)
    print("Date:", date.date())
    print("=" * 50)

    print("Bitcoin:", not btc[btc["Date"] == date].empty)
    print("Ethereum:", not eth[eth["Date"] == date].empty)
    print("Google Trends:", not google_trends[google_trends["Date"] == date].empty)
    print("Fear & Greed:", not fear_greed[fear_greed["Date"] == date].empty)
    print()
    

Date: 2022-05-23
Bitcoin: False
Ethereum: False
Google Trends: True
Fear & Greed: True

Date: 2024-10-26
Bitcoin: True
Ethereum: True
Google Trends: True
Fear & Greed: False



### Interpretation

The investigation confirms that the missing dates originate from the original source datasets rather than the data integration process. The date **23 May 2022** is unavailable in both the Bitcoin and Ethereum datasets, whereas **26 October 2024** is unavailable in the Fear & Greed dataset. Since an inner join was used to merge the datasets, these dates were automatically excluded from the final merged dataset. As only two dates are affected across the entire study period, the impact on the subsequent analysis and forecasting is considered minimal.

### Check for Duplicate Dates

The merged dataset is checked for duplicate dates to ensure that each observation represents a unique trading day. Duplicate dates can affect time-series analysis and forecasting by introducing repeated observations.

In [17]:
# Check duplicate dates

duplicate_dates = merged_data["Date"].duplicated().sum()

print("=" * 50)
print("Duplicate Dates")
print("=" * 50)
print("Number of Duplicate Dates:", duplicate_dates)

Duplicate Dates
Number of Duplicate Dates: 0


### Interpretation

No duplicate dates were identified in the merged dataset. This confirms that each trading day is represented by a single observation, making the dataset suitable for time-series analysis.

### Create merged dataset quality summary

In [18]:
# Create merged dataset quality summary

data_quality_summary = pd.DataFrame({
    "Dataset": ["Merged Dataset"],
    "Rows": [merged_data.shape[0]],
    "Columns": [merged_data.shape[1]],
    "Start Date": [merged_data["Date"].min().date()],
    "End Date": [merged_data["Date"].max().date()],
    "Missing Values": [merged_data.isnull().sum().sum()],
    "Missing Dates": [len(missing_dates)],
    "Duplicate Dates": [merged_data["Date"].duplicated().sum()]
})

display(data_quality_summary)

# Save summary

# Save merged data quality summary
quality_summary_path = (
    TABLES_DIR / "merged_data_quality_summary.csv"
)

data_quality_summary.to_csv(
    quality_summary_path,
    index=False
)

print(f"Saved data quality summary: {quality_summary_path}")

,Dataset,Rows,Columns,Start Date,End Date,Missing Values,Missing Dates,Duplicate Dates
0,Merged Dataset,1799,11,2021-06-01,2026-05-06,18,2,0


Saved data quality summary: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/data_preparation/tables/merged_data_quality_summary.csv


### Interpretation

The final data quality assessment shows that the merged dataset contains 1,799 observations and 11 variables (12 after sentiment encoding), covering 1 June 2021 to 6 May 2026. Eighteen missing values remain, confined to the repaired volume columns as documented above; these are handled explicitly at the feature-engineering stage rather than silently. No duplicate dates were identified.

## Data Transformation

### Encode the Market Sentiment Variable

The market sentiment categories are converted into numerical values to enable their use as predictor variables in machine learning and deep learning models. The original categorical variable is retained to preserve interpretability.

In [19]:
# Encode market sentiment

sentiment_mapping = {
    "Extreme Fear": 0,
    "Fear": 1,
    "Neutral": 2,
    "Greed": 3,
    "Extreme Greed": 4
}

merged_data["Market_Sentiment_Encoded"] = (
    merged_data["Market_Sentiment"].map(sentiment_mapping)
)

display(
    merged_data[
        ["Market_Sentiment", "Market_Sentiment_Encoded"]
    ].head()
)

,Market_Sentiment,Market_Sentiment_Encoded
0,Extreme Fear,0
1,Extreme Fear,0
2,Extreme Fear,0
3,Fear,1
4,Extreme Fear,0


### Interpretation

The market sentiment categories were successfully encoded into numerical values while retaining the original categorical variable. This enables the sentiment information to be incorporated into forecasting models that require numerical inputs.

### Verify Data Types

The data types of all variables are verified to ensure that each variable is stored in an appropriate format for statistical analysis and forecasting.

In [20]:
print("=" * 50)
print("Merged Dataset Data Types")
print("=" * 50)

print(merged_data.dtypes)

Merged Dataset Data Types
Date                        datetime64[us]
Bitcoin_Price                      float64
Bitcoin_Market_Cap                 float64
Bitcoin_Volume                     float64
Ethereum_Price                     float64
Ethereum_Market_Cap                float64
Ethereum_Volume                    float64
Bitcoin_Search_Interest              int64
Ethereum_Search_Interest             int64
Fear_Greed_Index                     int64
Market_Sentiment                       str
Market_Sentiment_Encoded             int64
dtype: object


### Interpretation

The data type verification confirms that all variables are stored in appropriate formats for analysis. The **Date** variable is stored as a datetime object, cryptocurrency prices, market capitalisation, and trading volumes are stored as floating-point numbers, while the Google Trends scores, Fear & Greed Index, and encoded market sentiment are stored as integers. The original market sentiment variable is retained as a text field to preserve its descriptive meaning. Overall, the dataset is correctly formatted and ready for exploratory data analysis and forecasting.

### Validate Variable Ranges

The numerical variables are examined to ensure that their values fall within reasonable ranges and that no invalid values remain after preprocessing.

In [21]:
range_summary = pd.DataFrame({
    "Variable": [
        "Bitcoin_Price",
        "Ethereum_Price",
        "Bitcoin_Market_Cap",
        "Ethereum_Market_Cap",
        "Bitcoin_Volume",
        "Ethereum_Volume",
        "Bitcoin_Search_Interest",
        "Ethereum_Search_Interest",
        "Fear_Greed_Index",
        "Market_Sentiment_Encoded"
    ]
})

range_summary["Minimum"] = [
    merged_data[v].min() for v in range_summary["Variable"]
]
range_summary["Maximum"] = [
    merged_data[v].max() for v in range_summary["Variable"]
]

display(range_summary)

,Variable,Minimum,Maximum
0,Bitcoin_Price,1.574244e+04,1.247735e+05
1,Ethereum_Price,9.952526e+02,4.829226e+03
2,Bitcoin_Market_Cap,3.019638e+11,2.486073e+12
3,Ethereum_Market_Cap,1.201489e+11,5.828170e+11
4,Bitcoin_Volume,4.048064e+08,4.293082e+11
5,Ethereum_Volume,1.260718e+09,9.658873e+10
6,Bitcoin_Search_Interest,8.000000e+00,1.000000e+02
7,Ethereum_Search_Interest,4.000000e+00,1.000000e+02
8,Fear_Greed_Index,5.000000e+00,9.400000e+01
9,Market_Sentiment_Encoded,0.000000e+00,4.000000e+00


### Interpretation

The variable range validation confirms that all numerical variables fall within expected and valid limits. Cryptocurrency prices and market-related variables contain positive values, the Google Trends search interest scores are within the expected range of **0–100**, the Fear & Greed Index ranges from **5 to 94**, and the encoded market sentiment values range from **0 to 4**. No invalid or unexpected values were identified, indicating that the dataset is suitable for subsequent exploratory data analysis and forecasting.

### Time-Series Preparation


### Sort the Final Dataset

The merged dataset is sorted in chronological order to ensure that observations follow the correct temporal sequence required for time-series analysis and forecasting.

In [22]:
# Sort merged dataset by date

merged_data = merged_data.sort_values("Date").reset_index(drop=True)

# Display first and last five observations

display(merged_data[["Date"]].head())

display(merged_data[["Date"]].tail())

,Date
0,2021-06-01
1,2021-06-02
2,2021-06-03
3,2021-06-04
4,2021-06-05


,Date
1794,2026-05-02
1795,2026-05-03
1796,2026-05-04
1797,2026-05-05
1798,2026-05-06


### Interpretation

The merged dataset has been successfully sorted in chronological order. This ensures that the observations are arranged correctly for subsequent exploratory analysis, feature engineering, and forecasting.

### Verify Daily Frequency

The chronological sequence of the merged dataset is verified to confirm that the observations follow a regular daily frequency throughout the study period.

In [23]:
# Check the frequency between consecutive dates

date_frequency = merged_data["Date"].diff().value_counts().sort_index()

print("=" * 50)
print("Daily Frequency Check")
print("=" * 50)

display(date_frequency)

Daily Frequency Check


Date
1 days    1796
2 days       2
Name: count, dtype: int64

### Interpretation

The frequency check confirms that the merged dataset predominantly consists of daily observations. The small number of two-day intervals corresponds to the two previously identified missing calendar dates in the original source datasets. Apart from these minor gaps, the time series maintains a regular daily frequency and is suitable for exploratory analysis and forecasting.

## Final Preprocessed Dataset

The following output presents the final dimensions and date range of the preprocessed dataset after completing all data cleaning, integration, quality assessment, transformation, and time-series preparation steps.

In [24]:
print("=" * 50)
print("Final Preprocessed Dataset")
print("=" * 50)

print(f"Rows: {merged_data.shape[0]}")
print(f"Columns: {merged_data.shape[1]}")
print(f"Date Range: {merged_data['Date'].min().date()} to {merged_data['Date'].max().date()}")

display(merged_data.head())

Final Preprocessed Dataset
Rows: 1799
Columns: 12
Date Range: 2021-06-01 to 2026-05-06


,Date,Bitcoin_Price,Bitcoin_Market_Cap,Bitcoin_Volume,Ethereum_Price,Ethereum_Market_Cap,Ethereum_Volume,Bitcoin_Search_Interest,Ethereum_Search_Interest,Fear_Greed_Index,Market_Sentiment,Market_Sentiment_Encoded
0,2021-06-01,36680.068202,6.887264e+11,3.353691e+10,2632.656600,3.064888e+11,3.989865e+10,32,34,20,Extreme Fear,0
1,2021-06-02,37685.717982,7.045313e+11,3.124069e+10,2717.154037,3.153133e+11,3.914886e+10,31,33,23,Extreme Fear,0
2,2021-06-03,39151.316184,7.331507e+11,3.339672e+10,2858.276702,3.339646e+11,3.810466e+10,31,33,24,Extreme Fear,0
3,2021-06-04,36938.720311,6.917475e+11,3.934549e+10,2694.497667,3.129911e+11,3.999658e+10,35,31,27,Fear,1
4,2021-06-05,35546.985965,6.629135e+11,3.406817e+10,2624.768915,3.027434e+11,3.473198e+10,31,28,24,Extreme Fear,0


### Interpretation

The final preprocessed dataset contains **1,799 daily observations** and **12 variables**, covering the study period from **1 June 2021** to **6 May 2026**. The dataset combines cryptocurrency market data, Google Trends search interest, the Crypto Fear & Greed Index, and the encoded market sentiment variable into a single integrated time-series dataset.

All preprocessing tasks were completed successfully, including data cleaning, date standardisation, dataset integration, quality validation, categorical encoding, and chronological ordering. Consequently, the dataset is clean, consistent, and ready for the next stage of the research, which involves exploratory data analysis, correlation analysis, Granger causality testing, and predictive modelling.

### Save the Final Preprocessed Dataset

The final preprocessed dataset is saved for use in the subsequent exploratory data analysis, feature engineering, and forecasting stages.

In [25]:
# ==========================================================
# Save Final Processed Dataset
# ==========================================================

merged_output_path = (
    PROCESSED_DATA_DIR / "merged_dataset.csv"
)

merged_data.to_csv(
    merged_output_path,
    index=False
)

if not merged_output_path.exists():
    raise IOError(
        f"Dataset was not saved: {merged_output_path}"
    )

print("Final preprocessed dataset saved successfully.")
print(f"Location: {merged_output_path}")
print(f"Rows: {merged_data.shape[0]:,}")
print(f"Columns: {merged_data.shape[1]}")

Final preprocessed dataset saved successfully.
Location: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/merged_dataset.csv
Rows: 1,799
Columns: 12


## Data Preprocessing Summary

The data preprocessing stage successfully prepared the collected datasets for subsequent analysis and forecasting. Initially, the Bitcoin, Ethereum, Google Trends, and Crypto Fear & Greed datasets were cleaned by converting all date variables to a consistent datetime format, removing timezone information where required, standardising column names, and filtering each dataset to the common study period from **1 June 2021** to **6 May 2026**.

The cleaned datasets were then integrated using an inner join on the common Date attribute to produce a unified multivariate time-series dataset. Following integration, a small number of implausible trading-volume values (9 per asset, 1–9 April 2026) were identified and set to missing; no other missing values or duplicate dates were found. Although two calendar dates were absent from the final dataset, further investigation showed that these missing dates originated from the original source datasets rather than the preprocessing or integration process. As only two observations were affected across the entire study period, their impact on the subsequent analysis is considered negligible.

To prepare the data for machine learning models, the categorical **Market Sentiment** variable was encoded into an ordinal numerical representation while retaining the original categorical labels for interpretation. In addition, all variables were verified to have the correct data types and valid numerical ranges to ensure data integrity.

Finally, the merged dataset was sorted chronologically, its daily time-series structure was verified, and the final preprocessed dataset was saved for use in the subsequent stages of the research.

Overall, the preprocessing process produced a clean, consistent, and reliable multivariate time-series dataset comprising **1,799 daily observations** and **12 variables**. The dataset is therefore suitable for exploratory data analysis, correlation analysis, Granger causality analysis, feature engineering, and the development and evaluation of forecasting models.